# 五帧动作头重排六条已存轨迹

新建私有 Kaggle Notebook 后导入本文件，将 Accelerator 设为 None（CPU），Internet 打开。在 Add Input 中挂载原实验输出中的 `joint_actions_all.zip`，保持原文件名。如果已经私有挂载 `pinns_matched_histories.npz`，会校验后复用；否则只从冻结 PINNS 数据源取得小型标签和标定。研究准则已经在查看本次重排分数前登记。[原联合动作输入作业](https://www.kaggle.com/code/pengjy25/fork-of-shortpose-matched-10-frame-pose-fusion-cpu?scriptVersionId=373610725)

此作业根据五帧历史、原模式概率和动作边际概率选择一条轨迹；完整包自带合成测试，随后运行身份检查和真实评分，不训练或加载模型进行前向。主要比较为 detached 臂重排与原首选的四地点等权 ADE，另列 FDE、事件召回/误触与直接匀速。事件得分是决策规则，未作联合概率校准。

输出为 `/kaggle/working/action_rerank/action_consistency_rerank.zip`，其中 paired_private.npz 含逐窗身份与误差，保存在私有输出。

In [ ]:
import base64, hashlib, os, subprocess, sys, tempfile, zlib
from pathlib import Path
if not Path('/kaggle/working').is_dir() or not Path('/kaggle/input').is_dir():
    raise RuntimeError('请在 Kaggle 私有 Notebook 运行')
packed = (
    'eNqde2nXqkqy5vfzK7bboRgO+zALpTjhLAqOqHddkRkcQCYRp9/evnufU7dudfVd3f2upStNMiIzIjIz4tH3yX9rxrGqO99ix/xm'
    '3twodj37m+sZ7tU1EvX07eC7Xoyoeuz6XvRtpNr2yfzmJ/Elib/pvherrvcl8XOY8ucwRT2dftzdy4/f3PPFD+NvmhqZNPn7N0eN'
    'nJOr/f7Nj37/FiXaJfR1M/pqZ5+32DxfLPdk/v7t/hn0mxX6528XNf6S+PanIunz8bfQ92PuqwX8JfHjfDS+2sAlNC33xn2/uN7X'
    'Mn4uRwnNUPWOyncQ/C00Lz73Jf/H98hPQt38Xvn21felwA2BrwFX7jttGGXGKqukVS5bGmYwGI0SFkXodFljdBylUIowWYv9/tt/'
    '2fAjTDzgP77bbvz99+8I//33KA6BL9Xg798/LvrZG3z/z991x9SP3CJMTPD/UtoyY935Jf55QwzzEjsc9mk7cXyJ/v7HHx8pJ9F+'
    '6P75j8shyzIj+mPpfRQkepyEpoHMnY/rJD8yEenz8cfPST52/v+s5afEJ/r/vJyv7fNvFKpRZH4i9k96fz5Vfu2d/9MEHzXIRQ0j'
    '8/Oo32m2Px6LzVv8S+ePz0D3AoAc9xn2m+WfDDP8M5pqYnzUVb796vxHOC9qdvJVg3v8TZqJC5EXhR9n429//wb8jaFMpkxQRNnU'
    'UNaiUEZjKMOiScMkcMJADZYlcYJWaZLGKE3HyyhhYBRm6qyloThmEn/7/dvfdMTDxMXCsEt0o1+6T1s+Ew3bvFvKTW7Be7XprxBX'
    '7lNRsftir2R5fOK7erV1m9pgatuj+2m0xm+jXV/m9lNU36UD9BJESa5hum+rYNuUmd6IUf0RaMsxGq9stbGSBveVtyv4euWV5Td4'
    '0z3CwOSmv5CZu0ebJz5SuMXNPR0yGuPFXapVt4TllTaaR7FdYqdjL7uKpnXKEArXjuznZQhMNJdy8M4GzXXIvImvXgxK1+f13Llx'
    'bK6ORpesVRpybW141zjtw++NKcS9GXBsK7PzuW4ecJPztNuTua7I6mhqo9RSytWXt1vfkry97InbkXuC17khs5nvg7abNBfl8wGY'
    'zPyDdMydVzevBkpyP5yd4VOpx/SeKTDO4TVPIV2lsQpYC2kmQb8sbe7F7cbpyT274VRvnkHQ7Kx0Pi6T7lEtyHe22h13KnmLfkhI'
    '2ll4y6eDQAd8rR1a0+XBP2udMEvO0cfIxqqStQXYWs8WoboNcsFwkRtv4wfqL405lcezyghtKNNhec8gy3rtskQyr7Wa2i4A8ZNd'
    '0VkcV51cQHT6pQXko9zFrDxwwtS09rTkuJNcvr6ezmWmFW1aoBE6xW4+bU3ltdo7um+wTqPL/A2cwcM0OU4zzLbpXjvfLm7517xZ'
    'QN9m9f0Y7uuVk+udNsNmU1Hdc+Zt1nW+MWLHlYM3k/TerkKURbJY0tOr73Q2QPYakxx03NeLCWlPuq/clZrykMFriCG5O1p3NonI'
    'LW0cMGSuYzxtPNaR0/15N6xY3KbIrWiJEV08RsKLzJWutlCcBZNtbrjeqzdh7r9LHBFspnWf3S77kByJU6w4Wdi7cuYifF+nkQvR'
    'XVvoOXnn2yP2Ml6fp9l9XOXSakmNfb8DoKfi4RyM3ueKolTE7Lk/NrrBXNDnm+fcRAqg5maM3ZzI2gLEjMKeqIzv6fFqqJa5W1H+'
    'nlRTSSMDuCIfK69+33W8aTZ8SgE5EDYAHdGVukar+tNMyvPmrTY7HOHCA5mynh1qQ35ehhNF6hTf7ljo+ho21CoOjm9j8WVPu/UO'
    'q7VmYqe9eoxoCXJnfDavq+JxTb4vuXftfdh04vo7ZoR+f+dUSs2y2lBenbFelVxctFcbdXgpZEGUkW9NM3phnnSNwUlKHqP7XoIJ'
    '/PhsvsNDeu4KRfB+VkxucOUDI+eF9AIyIGF9wyzBG+B77GkT+aFa2SZMqcKMw+3qkrsHSHZbzw9Pd0wMSKIfd8FDnseuwb60WpAY'
    'XHCK1Z1dqQlzSFNvrL5eIYV1c37zuc14uzb8JbzFniVzcSvC8hSI89GpZOXvh0FPNbenXdvqlAamV1yYtGk2egjQwn3ueja0Yz6c'
    'GNDNtDVcryUqNt/XdmbHnkaJYYSTkjtbtyKEnsMBQZtFrrhE1WKtIo+xIRKcDknFFXOZ9+wy0yV0J3F0GTbb9lEiG7rWWMVNTZoJ'
    'k5ZFiJw66mhCbALOKUzDfWfbwK8PeVziMW0s0tt0E08mk3VbRNvRBEAao3oDeBnDIdlVD8dGRHe3Az+WFngmtBdNLoiKjR4Y3cqT'
    '4xDw2zQbQa80OJHcdjU81XIIBRmV8w4xyxAcwzf9PE3AC2qnMpKNtlD5MZAzCSpm4NBYC836YTQtzoX+oXOTmls8IZj19Dm61Hut'
    '0sm3g8ah71rDcgS1p+ha3OZXe9xl5cSZMaCvHkVU4PzN0O77tfVlYsLMaN7FzgNEog7HciO5LbHqLFm3mytG6kOrDhS28Xi7GtDh'
    'DFlX+igbwBoGZ2kheN2Lhefw0u9qt5s26cornXhZFdfaeZn3wARVIIyKzfW2IWb2RPQ8jOV+eTiNeZ2YPvLHOxyyXuLx2WoG914i'
    'cvCADSu6+8rUJB9pQnUpqXR01/37ITfJhBXwVoVVAdd7PgPGgTdpedMZRGvmohgiYbas30fbZk1rip3AvbUtfcfnyseI+Ew8veCq'
    '0txlTz+5vP0g8ELSC2bzYukliKS1F57LQ4HpwMyrCUrutlxYkif0sObx6LjY5ge2vhswYDbst6MLebwW3ELbaQ94VV37y3NJEZdN'
    'OLfXLoBXfN5aEq8PXekpBE9Bz6GXbQ3ATbAo23R+MV7d1miOiCyBipr3stXdNIZGG3fEZz5r8GKKFLxdUNyOfKexuDGlOX85IbOQ'
    'J3ezM4SCdydnHL1jdDGrw6MxmUgrmc+G46GnK7VahGwy2e0Xd/FIjye5XYV9cnaNPFMBRipq6OabQlPiAGpUQ2rUkD0XBG7IuGIx'
    '7TamyMbXCaJhNmS2b12LonHRnBb/uQlm3dGj1/TPTu4VQRqvntj+Vl40VktgtszdlwOFVStP4Xp9RlC602eLKD7JFTwa3RyMqKAu'
    'Vt8iJRmxRr4HnZvceFJLbnNX1YXNWsidJ56moqLfqBCJiUIAXJ8h75Fark3qhjtoHTk19IsFZ1M43GY2A+ckv8Tfpgvms2lW/kkb'
    'AqPa5yzxkXfNC0CrP9dipCB0dsfWtYxN6W7tBIduKyvYVCWvr4ZZM/VKgi+QYAU9jlni4O9uOAIOilfc0cZU7WUfuRXqCcCsQO3P'
    'TjfshSPryttiY6U5uil0wT04OQRspVAajnMSvO+Gm8hFJWJCDc9ZRDas9qGVMTkrFeeAXr7DFj3uPvMTbrULkP4z2LTWwShxRlVi'
    'NvSO6WFG76GTP63WZslEcp7FOjt7lnvNIjl456SW2TlczgwM9IVP0XISKpIQ32oduX1vuwees8vauQPXCS0m363Ua5Bznu3lrpvD'
    'DlWYj58Ph7aMPPQFQfUgulnNRgt4e7eNXLxUmshTqKtTqYV90lhlQFaC3PVpHn2MMo+Xc9c7AEQRLVuvDZeyUz9Xo0701CnNHqOX'
    'cgzOIVYvIvvs5pW1CVo4WOtKsJnkBrfZfdPaQYl7BkSpzafTiJ1vLg98dF0Wz1J/u1JrUcQeXhubLutsf53jNwdt93qmvFhs3R/k'
    'eFoiBvUZC/ahYHdNXWC1cbpl9lZspoi/uPLVMTuOtT6BhwEyl3Bay6alDHo0CizGXFAN54sA5thvcT3zXtqsdbdlKKvkK+G+HBfT'
    'Sm8p71XDaMkEPBVHSq4+u6j4xhZ2xOsCEAe2jpTN2pGwJ8EypDZvsfx0m2OW9o8eqHfT045ufYI7ROctsD2EB8qdbpPLh+65Rd9J'
    'c/uGwsHavnFAbzFrveRzc29d2ykDbmr3HjEY0NUNttPlOUIHbanUVFywWb4+0+5hs0M3254iuafVZpHl8Xi+uj2caDB/2+dVtjgz'
    'a6NGk13pjNDqNiXpND8Z7KQ8dPBWGIFbfwM/tXxknkw99sMfl+wnPqAYjdJoC0Nxwyxjn5zH6gyjfpCfjpKs+hGjNMKkyh/ooNEG'
    'q+OsZaqk/sGLGKrrGsb8wgd+hiAVo3ihpAjSG9p6DYOxCbkdcDXvIZ4wKy9waA6VT2ZN5g22FC879Iyb+aVTVnnfjzdueB2dbHxt'
    'So3suj6vJUavjeS1ctKso3tkcsehcX53VIKe4HebPdytBCvPvCfGPZf9ayAfmvrt9NaLywNVBNn+UZk09nhPxuuNNiPpdhDfX+1C'
    'F4W12XnCTkuTrTmvlMeVJeEobw+0A91dmhCBYBVzAF5Es/QycXBn5rvYnKyRArk4az0PPD5Ipbrs9WoK80EWoDZMynZnganj2rGT'
    'I66b+FxA8/4EL1hHrtgGuM2TM2nFmRyfADLw07n3yt71asFoIfZd3OM2dqJVd7e+Ss5mpGmO5e57Je96H+aeYa8Khfs3pAUkH9Xv'
    'gzl0LVehF7c+lR9cfs2rYIcoyNHId6++quanVN7RKl3nUzQT/pWBqVEhJ+u5uH9WCesSn4Kp7yWv+FW8XwaVNpvdC8d8sG/fWgZM'
    'o09euMStY1EewvnuQFAedld2Uri2Ld+Gjz5C7z1RQFKw86q0j9u32WoheYwWXKhXg53SHVmAz9MMALPx4zqDawJliAG7rDdmu+M4'
    '134wm4h+0hxn6Cujw1JnO4DcxiVwsqI/r5yHtFItF30Woov3QVCV/asydT8Xxfho99Xy+Dk0mvmlj1b2HWxvFleJlkrYwjanRh1o'
    'vQpBHilm47dOcHttwXdBCy6gGtnL35ErPLgR0HgrhzLrFwC0mwHs6pqbce6dl1NIQKFywal2cL42guo6laMsUyH4U5RyrQfQ3PGV'
    'e8nieq/xXowPTgUTfTARJ7i1Tiqx3IupKcVbFzRMO3uoOFEGqN0wmLJDl7nNGi9il/IVXwLcHlgezMN6N9hcTetFSZaz5xKolV/y'
    '/dO+XCdftdFFFf3kyHELB79Nr3shYIdUipnmrmDfF/qe6Kz6IbPWpbQ26Yxh/gyHM4kaemoIjdv1xcOyG7tiFSkYC9F1St3Cg1o5'
    'p9wE2FcWUTWoz6VOaTjTyUmjdh0x0gzqLTNvptjlAReDs163BjVS/SXB+hVbOPAzxKc3TfSFsldzb3us8lrOe3C/nc5SlryNNDMc'
    'dptF2I2ixQrenY1jl++7pHlgD7ltKgdYjqyxZfXCwdbnPo79Rbr39vCdk9z2Kffa5aWgdEmMjfaJ6KBSHUEvKBds3zeBOvkQ0H3f'
    '4cMlXD/6rq7Zm6zf4cYtW07m2G4qR3XfcN/SpJtrjVcD/Im9Td/oQG52uBCT/QFqejDPEu99h4VmINbsBny/dZdfU+A2MHO9Wkdf'
    '8/0+wrP9aOOcnRl1rtBtujFhB+kGF0RDKA25Uq21/fi7taEDRu7V0cQgVz6Ac4/U7YYx3qy9B+0BEtbVuelBne3CT3cYUpfnPbIj'
    'CkFeLh7EkczfqOGjV85bU1IJVlPrU3sutP4QKpQqJf8t3gbcsTDlf17AYfLXN3V/XcEGRpokppUZCld1CtUoxkR11MQNyzQ+VzBq'
    'YgRDEgZNMTRJWChqkCpZpj9XsMmQZbX851c0l8qklt3p7DRCPpibvhhqISzgDWOx3p5zs7ikJsMMyMPR6j0Ogdx+Oihj/TcxpRwi'
    '/6kXKorbLYYTm3p3s06VMzu8hKDjG1jllsVN7PHyazK5ij7SEAbPudKAaqvngVCWs9oULAS3D/YXqYUcbo7d/GVttxcdym5NVyuj'
    'cbax2TZfbq+SxT4T+2Xvk5r9FSc2nXzcMJ1y5STAy6p7QzAWme7LV+3aX8Onot/hqjVnXts+T7lOstk83d0Ie54N7H2rjgfmIlTu'
    'rMdX0kPurC4LN2YJ9cC1+ZS4Cmy3bPzunESwUZDyLzKEN+qovxpCyxHijC9BeQdTx20L3dAFJyIJ2QQbbtwha5tbV7zfR1OWPSax'
    'uRyW+vetOfgUaZZiIu7JHeLmvVoJ6z39dqgjUr+4zecTYidXh2masJtdPL1afSUFJQGS7EMi0Ta0IXZ1frHejF3kAkWxRc/2tRvY'
    'rF+H1HtwfvRJEB+Ua3NHkYIsLY5b75VlUwp8pdScst3tNv3to/EcpEmeoU4QkNYbwYGa4I1LdbCymZJY1O12K9+4XEkhjeS7emCU'
    'NeqIjeqknq+jcCePQx3meO4AyemJOeXx2Ncb3npEi7n65+g0oJx/2pnPQq7DV7Tb9tIV/dVmyWldTqgSVoYbPmJP/NtllBy9U7sn'
    'DMJ7AplQtCBpdsbj/Si/0xdvzCJVLG2MR7Q8J1rN2UKPnm79IbEXojsYFSTMhZaDIQ9Xm6etocUaWE73sGHbI+Us5zniOjwBs3Za'
    'W+bgq/jWuG69jechbhkkeIGQ5fVzJW0QLdPLfk4yuVy+kNs2ZWYqpuVFu994pHZjMJiXaYRms5FksO2O2Ifwc9g/WUWgdPVzk+oz'
    'WUnjMSkdnQHSJeevt9ww+p1q06Xny8ji9GLbRTqBTo6JRuswxK6l6lvZXh+bXaHhVOps4VIJFgIzjkwLBFMJss8gvd/NT0m5iWUN'
    '+xKKbaXUX3cIjDrfN0i7OXvFSw0olpxOoTaYkI1i47m5MSNvycYJf1gE4jLGt6NptfmMyKLwHCTPPSSwlKSW8sCRo0j1BJQIytuX'
    'iZTePcXl0r3o7Vgjzux4HnrP1nx5Xszl086Yz/XVisyLhQGpUv23uG9vSbwGxAuxzAtVgevPj+z9VsTaRetRWPf3w9lUrxTjcIPS'
    '9jhdsWkJ7eNGZ/+A6enYn0lZVOLNg3Yv3wev3II/ptGFt+vr5RygBGuQJ7NReegbE4SaoVxzprT0VDpzca10bVzwUpabXLvL53HN'
    'rDltfaftjUxkxYPO768oMUf4zxrPPIFfGx2hsIOfSU4RcyWoldE0/9hLtdGoQS9zdYLdUes12hDHMAaf8Eua1AmxMjiRj1oOk64R'
    'Z9/HuG7vN5GoU693JW81AbaC7eLz4zh5HxDFeR6EvjuQBl4/TxtVsFs9XT+QMKv273xTj7CCAp1NsAOz5TcNtrQPOGmsk3f/GbOK'
    'L9yxU4OXoOHosKY+lQzXPTQaEDIykuIJWaFggS50bGg0nw/8ojQnc9iIqV9c4dapboeAIdMBm+O3VdDUT4iubBMHD5RS94SJe5pe'
    'MwKfr8wO/IrN64+HFgxGag18i0qzDvO9ohg0m50BiCxhrL1O4FVrvE83BxgRmE81Atl2+X1b9p1RcMEYM/mEwd473C31CKcZEYyD'
    '91ttrZOp71b+Db4xcu0VhvDqGW/tSbtbt2ZdOdgJc6XWl4DSJjW0Z7XSge5v3Cir6KP/OOS45+LyzszZmVuwlQtfrHrLzzGYdTa1'
    'a1frj9ZlWav3lPqsYE9CzOwPOxnNDQ7ijh0AJ35N1C2HogeKLi5ygQmG22JlfVgv3Ye9rpbAM/6Y5DcftK2mhL8WFyddl9+xVR/f'
    '1KxhrXvKfZk7y8jx4djBqjB5ECf/scUay9qgt97yw65buD871Y409Cde91MNyddmPMAQkBP1XtNTc1bD2PUfz3IzQGWudTX166fo'
    '7yGiW6vWs4NYSHGZGlNRehupOmr1blPW67V8eVo9CyG77F0lBR7DeYc5z6Vqt0DwtuYC00oh7DfQcoPRXozH2U/gnlz009AcqW2F'
    'fRQWFHxuywWl+LQ0U5gMzGJYf8IsLpZmAbYrdHsYXW9FhTG5HV27YFONF9VwdGIq297mA4cuiizinFYqCp8j4/Sk7NqTqXkJKTD3'
    '7v4Gu2d3TdAja8eNPBRJnxr9mIljjEjM1icX91hyIAiELzmwbxaj2Si3YZU4DzyWysfDaV6nyLHF9zg2YQD9WVkWamjRmAtcC0bk'
    '7AV6cDYLybDmrs1ivCm2x+0O5s/BmtgEj9VkKSxeXNjVsR1uTPvzNv/WB/UjOD4WvcxzeHuxanQPTeIwlrRziYy14eIU2kIpqUVO'
    'Q2C2e6WoBe1Helua2ywzxkqJtid5AIcpH95sprnZ8qK9IYZ0uL6w6X9ykfrQWvwegbbiJg56QDWqLei5uwHZouGfaDdZtXBw1kS6'
    'k2kbKJVvtzuKskrRV6q1CK5WxaB8PV/xKiwD7qBLJucpcVYIVErccavTVdXwUFabMl/fVrkRou/IctNRGzh1h2bGfV0ckoNd5i/T'
    'qMBXVcqryuys2tJfs4OoNp6NyiGnrNfRHg8PwpuoJOAYxU9A+zae5deLYfKJAxZ4PlytHIdszldOYRav+cW8g9njzJou4Os5a7cp'
    '9zmh7q7VD4vshgRKcLvVs6Bhi5rvr0G9b2eQBQBPp1it23pbPjP9UHhQSzMHPduESbSY1balRmaduPBuVwaKPRv/XIvBPBmCjtg0'
    'GEgcyi5VLCm3AZ8sn0jC3fv36bPbhe94l4KJS2/6gNKEKJ7Kzx5Et419HDfqzZXhMKj/zuebS5uTz8PF0H5eIjB2Bx1WqQ9l3m/6'
    'uBvUTKq0WiGfmv1+bLlXDx8P6FaV2hNvv8i+xGwgThGlqkDh6SwW1cDghnW6nXRGLt3wHwECpq3wPA668+Pxji2nE90qLQ8KDUcs'
    'eq9seDm45ONaMjHazYoJ90p5q3Wul1eJv1y1/fpbW4uHnXH71GYxV7W4DdZBhM7F2EuduDsE91HsXst5eB8CFWts33sFp9PruRYQ'
    'HCstcHVMtuL0kT8tWaQxQm5z/qjXuUFxfDCjFG+UzFanBFXXROwozrUK7dfmZHy6Fh+PKVgv+2xBiPgkPuJ0zzJvQu6uJmFxbNVa'
    'H5Blu053PQO5A99XOsRihDtPbTfe+GS8nLlzGe77TBUWNxKMSCfwlU3XjaswHji9602PeWH8AYz5Cal49+3W2FoMYb5PJVIuUOe3'
    'gEeFmaBlzb3q1rzzdGImwmVcxnKtxdFgMHE2e840+dDsNQ17MqqbYsu/wtj4Kq+LHmUUfMZ/H6cqoQDNq4mWbatjLgeKVGoNTsd3'
    'whKl0gDdQGSfmDo1Hs5GYu+wmefyEnNwJfyz89ru+DjdPqswCJfyceAlmwp3plp5eXu6qfOOY++kU3PlLc7Ljb6dX2Us3TOLBcSY'
    'FqCr1349m+fK73JwGy/WRNFA76Wjlz4QKE/GM6Onemq09q4avJ7DNWW2eG5vfezdbuZuM3trHjFgTfSgiTfjrPyElUFdaPLn0s6Q'
    'P65WX5rerAE00uqW6Ca3P9o2crmW5rlk/h49d++7IAlvudfYnvaD6l2QoToGLpJ+WtvNctv7ls7hPSYiNQFfowQY6ihyevf9vY+i'
    'Y8wI9hRxuAYpwHs2OqxpUL1fj4ML480lcXEf8ARiWvCzUJRc+GkiosAMT2Q+lm+nxLkXkEG9pI08Qia56va6bD9vWQNFu5PRdm7C'
    'PJKVQHI7J6olHZlRj65NymRtEW6hkNPu1HnfNnPMrVIeBRzcuoez4g2fsiUCU1+jjA6WvVMLaSLOmh4ND/kc8kx3Sy86R8EFWB6r'
    'bHXlsDixmR1S+lGUt5VCLRgtm1KpLXW94aZ0PfZwbJuV69UmrqNzS3HHwezSNBbVaXzLlEAerOXNMDiGGHPU3CHXURvlaWPc3rn5'
    '7C1EYweLKJZew+XJU7sb8PZ10uJTfsrQAFkXFCIyTrNJZelouZy+YeRlVk3ELnOnFK1bSDbnjJzOQhHaVI/H5VIrXx4wgQxS4QHg'
    'wbvPY1bUCk5Md93CBbpk0NP1JFsa4w9UuI3qechT2Dw2ZWlmfpsmubGoSB1jWpWPuVz1+QGxw7XEOMQ+XFfP8zJl78iVOV22F3Eh'
    '5HNNSekwBzNs1ZlzqI5K65I/tIbb0tWosxAWEvkb0l/Usa55lw6Z9hwedbSW7f06uDQkbhdKp6QnEVMR7G13nDBbJS9uVR3Ki7p9'
    'G/pwtCiHJ2c2i/WDCedGMij7hNi+ynm3RxwvpoVGYw0cbxunXVuJgJ3TupXwDCzf8nMpPb0cVnAnO7hTomjJ+KQHF+xv6vBewnLH'
    'wVFpD/zahVGQ3ME4hxvuClxnWelMsSsHXSnPTf8Yx4N1O7v2F0cAedDuo3UGHmi3tN8B2R3aYkFRHOUneDRul7oK2TFKox6FdyQ1'
    'z6Tz7AOCax9wqVykYCTyYGkrTUmzlSek/twpDnPhY3ZKtEtZjanKLMw3XpilbbRNt2ji/eyME1Zzf8VYDZngG2wy7GpET32uH4nS'
    'r1qjzNsIudaa2F2iAitDvJfcPOPSY/inZlSlzavYhO5ecymsiI72kHvesAJN5kfvU47xSKKu7Ru4P/XMR3sOrMfRPDzaKBnDysyZ'
    'a5prJ2J+2onGy6dbbusqeFfc6WOveVuLtKygymppPc2MZ51mjBckwMXYBnLJKB9FffUd9R/FVuqsqwCutCNg3lGEntTUzN2+c7HB'
    'bcvNU7r6kJyIS2h4ZHGvJsEWau8Z+7xh5q4gXii4V6dnuZxvPRAhClrmdkJfD8v8gakcUk2N6Gvdh3pBheXkpz3PabPuZqP0GEkb'
    'HMa7clfw9khDobrtUnV3VmvX1gFpbTbeg5pbF7z13CWq5o3CoLNJey0lg7a1FU+zxV0i5xbdJUisW/NBvhetJoODkr13jRMASR9k'
    '0mSuZLE7hQrqZHKH3Utv9soDoy635R3uZFzF4XCHWzjFU4Ge+cPZqlWQx+YDH9dwwGnV4uoGX3e8Q2cHY0TUedOT3FWN1Lg+fovC'
    'EBvHATQjsoVvKc1S4Lxro4EP30wudCx1eAzfBFxm1bpcGQfW7DGIKQUHzHznUhmI0bXE1ufW/l7SttVwz6doIT/bgxiXdAfdnUO+'
    'l7ur2ZnDSJVve6tuvgEMK9kqE+b8lmkqiLqgJJlgP1UF1PQnh3GSnQ9KuKyoLUifj6lR835QxBrk0PLaQtd54gDz90+q357MxoOM'
    'BFquXCKxVx1ftUq+p43xd19Kx340Mk4SGlBdMCrsvRG/koHDtDjYvJqD5oTkwJVsUVFuGSAefxO1ytvKqhs66kwsbLqSuIk9J30t'
    'qz5rg+mRMZ2t4CfsM0aYWKIx5JFMnh3owDkUOzsMtm1VuF7QEvuoXJWqb67bW21611uHG4wL+7M0SlEIyaTrDmm6ApfJyNL4QPl8'
    'eu5WBoPbrHin/ZTdD0koKgC5EKCy/X7IrIqPLjrvnN1+yoNpeyMuhr2ZU/Vrm/EHG4FFqdS/7lpOA99jldUjX93sxzvQY7zXYMnv'
    '40F5EgK7Zc/lJX1RcD1jhQBkHhKJOVQufdKaOoX2drBLxhG35WbjLANLzAxzVuDpcWxicgPje4FQTvVaMXk2u92h2UBFyKB3uVI3'
    'PNf2+KG5LAxz7i67VJ5hQU024qy9W49aUH0Xzevz2WbT1rpjn5vXvtDEjKnV4eauWAfv+ePeLo7OGjwOOzp43u3xFqVXlVxS6XFh'
    'gcvNkMbiXJ23SZLe73ZKSEyiwfwR9NGX1Jh1ts/4khqviJ821MWujg8a7qsvVZD1TWGivnElzESP0rjj4fn1kYbQZGrPfC5mznSl'
    'nbjswm5VEr82Q3j5qgQWZR+ABgPlNmhxRqf1exPLu6UMjBDT7p26zZyYP0vVWrzUuOFN3Gf0xssuyNicjrB6PCiua5o81DGyjuZ7'
    'i3zicPGw9kzhVCjnk3VeTweVfGtndc+pW5DJXtWFOueSFj6N/Tl9jORNq7fDmDvhprwvtlarLGQKk24Dyfc2x4TxlM6SrA857LJG'
    'bnnJdcQdgEHm7YIuN+abLOh9pdel02OvawE1I8/wlZruYal5CBoPoS4t1/1JfU/Ix1Zxf5wC+Cfr0JSdwuISeE2nIyIXluyIB47j'
    'cn9kOfzhlI6HA//M2/R5ir3uUba9bE61ZsXdS3Qwxxw/rXbGQAzG/aEGJHebv82OB56IWoXqw1SPU+JiLdfQpLo9rsLXqj5SmtId'
    'PFj5Zmghnl6Dh49rOZUaU/LMq315xHnDcJpBZTbNVBgsvGed+wl+Q3fTcvdnZAe+B0hR1DI2eqTWUd7edWxSDAdohbwuGq/jNd2I'
    'ZWF4ud0uGq/NKjnCMg1NOPe0w4YDyNtkq5dzd8Z+t2TomZ0K101rdl8ToHp2/A/yM2bpeCss/XO13VWu0zeVNyDwPnlpxXe3Kvdo'
    'e9Yl8dyZYVZzNXi3x2y+D1Iij8/mx+4pbe79XKuKFvRzM+eMWneS2g5nu9la5m83oyM0OabThMZLJsAJzn6Hsk+vTtTCR2a77F1K'
    'hpP54V4v9VpCA9vqrMRsl+gmzgHH20tLGsd8269+kr5V0UvFszfa8ZX5s6dtO8nQe7y5gFaYM1HeNsBB9Oh3nyPkIAZwjl7l31YR'
    '2UfThKmQQdpojMDLnD1l6EDUdvA06hO6vZRyzCWbHdnIjyfGCIEzttgV7772uU8k7NSwWyBK3dLJ6d5Z3QeVipmv17sY/9qWow0Z'
    'GU30VYt7NROaNLnLpyqCYctPQ6cpPhrwRGnUlp+kZcbYFHOEVzvuTzfbo+GvX1FDbJRKysYDbpfZfMksC5xALQ8kvfUF9uaz2WpO'
    'CU8uyhHuqpMTtgOQMjq+t9GaeidaV8Z4x19iCG9mg0Ffag3o7sZ6l09OjbOx58DPU8vw3Y7aJcqDiWkZqA3l/mtUCCNWEBeyeRmi'
    'pIfvToo8C8iBvnQiC+NyOGovh+ciNCnZ88pWHpSnnxgpcZcvPQpoVUeeyyMsFcwdM274KL+vdyjj5+80sRnFyr/+Wo7jmIERKK6x'
    'hoGalk7oKM1YmMWauKWrRlnTDRTDaR1lyzj7GWWyBm3QmMnqFqtZBP7nr+VvRRRv6xwlW8SGC6G0uxrADaNdCl/29JjrNAtxG0Zc'
    '02m2Tqt2HyyxBEQh1RexNbOcgE5lfdUGZs13+2G38iG4N4AJsvOGCNiD7mJSOC6VwCxT3HMg9T8HtJeS3jTwpWzpY7Ooubs8Og1o'
    't4psoHAqLqlt17H6RnV2M8t6kuu1m7smD5yu4FTnc0n+dlPtxSwqGMc75yo1AjhfT+N+QxgoDv0+xwWht9tM8mkBOn3gZx8U34jS'
    'Xnl1I+8X2gHWvifHolih5UJtqZlhRyp5i/t9Wo5K/vtVhKslmguGvn5qds6zyVHk7bt1qy/Oe99hUWe9IuMDewtgce29D0EO5OdJ'
    'bqQfHqUMUTqNdz461FuwphpCbRTedc4K0oftVB4+h4yFxmbeKdfdlredD/fXt5qWtqH1MMjs6JcWjW5jKpOHggydoovVJznJ8J0S'
    'FsratlX1sBY4bFyAF1bsMB8kq2oZBzHMVRaOV7nRzuLtQryFOkLnjY61MHH6BI8mbXL9EAOcD3BLPEe3acUllxawJE5QmX5QEwWC'
    'q/6FDMrmxMXqd3l3ugCtSw6ZLuNwpxeqyQ2/LHoTvlxYWXfKG7n113kMtNQgXsWl4y5Ql0AFf7HxoG8djNpTtSoQA5WdIJ1Fwjqn'
    '1UOn7ovEAGxeuo0d/Jpc0IfQaqa7ESWfJkafPuoFPWt/oDox8pD8K00/SWO4LOwWOMnsZLekNgVdatcmKKd/0FO5H+g3HI7HzRut'
    '7m0jpPtLf16uTtNX7YDGwzK2wYpFalEl3fPae1bNvBnm1Q3Qmpf5DtAQJX++y9opaiC10bkGAQprHdpDtM1f8WS67gu43Fu3u5MG'
    'QPcjVSy3H6dEMZqP06VEdzvEnaw7/JJDjEYNb1rHw/J5h7vzaK29F8BCTz/1BhZu2WE3hxPC7ZC/DdBjRfVql8IRYepzpkuY9SpV'
    '8su9KhmsjvXy2a+afjM3SdD6NiK33R3lV1KomAF39igNJEAWbgTdEFuOf1zFzUSsKAzdQULVoe0mK/UdHy+N2zYEbT8Z8+h8kOaO'
    'cQGwUjyn6zJ543uyaWBXRUckxfXEy7GLPwp8/+JUhuYUeeo9/7RZKkxeXDCkchxdO4VPxLVBfduT6dXzkVust/uuXShkQVwOI+G9'
    '9DiEldaGkSgpwe5aa/P0KAyYgwVV/Yb2vm+g3vuJyhsDhY9EBFwL9KIBFgrr7FIQcLoymIgYOqKncAvF66fGiljvXRUvk/ep1v3s'
    'v0ieh0qHD4yqBy/NaMkO1o1K4ag1M7JkoF10cy83wA9SXlW6Ym/CLRolKsTY9AL+DXz9ZvnhN089m78DPwkOUXL+PQK/ud63P8kH'
    'P9zYPEcA+Pffvn3+DDVWuS9+yw/D1P3zJTSjCPjFj/nxRUH4dBomEIHgz9F/kij+5M38iBwVp2jgSwf4wzFvhmt/buMvQsRfU/+U'
    'An6xIP74WhX4Iw0/C1C07HNx/5L8LcqiH1/Emh+u96UeQP+LgfF5aKqh7vzi13z/4/iT8POH612S+Dv4m2Fa334RZ4CfJn/qsE8e'
    'MI0/jQvNOAm9b555iwHg8u3LM5cvT0R++BkE/FL9I7RPvvZTHvz4yfp2+eFGyheNBwC/qZ7xr9ZefoSmavxpAPgvdv9jAb9PfM8E'
    'f/vJQ+L+XOL3f8tK+v77d40hTVVVDV1VNZLCGOOzVymdIT5pqUzSKIGaKEGzhMmwhmV+chihYZRp4Z+EpjMUUzY/nvis+6f2b270'
    '7Wvqv4eqG5nfuh8zJn7c9RPP6IShHwLf/4lg5Yeu7Xrq6dtVPbmfWJjGvydOfeL+F9XqL887bvRfdv2iN53V+BN2Q/l69NFsRj+8'
    'y/1j3ccqxkJZirVwxsINmlVpC6N1/WMMhlGqRuM4bqAabWoEzpZNw6DKFI1RlmkxmMaS+i/rvrT+w7if0f1JyIr00L3E0Y8oVm1T'
    '+XMhZqx+7ay/WFq/nv3V+1PW9Syf+4/vCPJX7y/Wz38fCfy56eLz5Y/oi7R08SPzLxaX/vHPZ0mmp2f/EPgOguB//maeIvPv/5hA'
    'N70oiX6p/7Lha4B35R4Q5Ec/Pi039L3fv0ubRV+cSM1F//vf/4l+JEqdSUtozpXJcqws+rNOsz3//vfv2PfXb34S/8uhSP3w6Hr2'
    'H/+NZvZxnn42uP/4OmPmzdSTWNVO5s/V/Hkqv/+3/3T5Dv4T/emr9XND/Gp+pgR/h74s+8//jZ/1P+j/1wrtO/ifv+upwX3N8fuX'
    'Lz6v/5H7BX0s+OJ2/RyD+N4p+/7/puFL/n8cfwk/Rn7Z98f3fxPeP53zdVbB3/4XikQbtQ=='
)
source = zlib.decompress(base64.b64decode(packed, validate=True))
assert hashlib.sha256(source).hexdigest() == 'de0b6e13a6d2ff7f76d55e1e437f28975fed1672c576ee358c3844be74617f43'
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OPENBLAS_NUM_THREADS': '1'}
with tempfile.TemporaryDirectory(prefix='action_rank_') as tmp:
    launcher = Path(tmp)/'kaggle_cell.py'
    launcher.write_bytes(source)
    view = Path('/kaggle/input')
    if not list(view.rglob('joint_actions_all.zip')):
        st = Path(tmp)/'inputs'; st.mkdir()
        b = next(view.rglob('joint_actions_all.bin')).read_bytes()
        assert hashlib.sha256(b).hexdigest() == 'b84eaaadcaab4518ded15c83e9d746030e03693e89dfe9e23b15ef2f1fc8587e'
        (st/'joint_actions_all.zip').write_bytes(b)
        for h in view.rglob('pinns_matched_histories.npz'): (st/'pinns_matched_histories.npz').write_bytes(h.read_bytes())
        view = st
    subprocess.run([sys.executable, '-u', '-c', "import builtins, pathlib, runpy, sys, types\nsource, root = pathlib.Path(sys.argv[1]), pathlib.Path(sys.argv[2])\nsys.argv = [str(source)]\nif root == pathlib.Path('/kaggle/input'):\n    runpy.run_path(str(source), run_name='__main__')\nelse:\n    proxy = types.ModuleType('pathlib')\n    proxy.Path = lambda *p: root if pathlib.Path(*p) == pathlib.Path('/kaggle/input') else pathlib.Path(*p)\n    orig_imp = builtins.__import__\n    def imp(name, g=None, l=None, fromlist=(), level=0):\n        if name == 'pathlib' and tuple(fromlist or ()) == ('Path',) and level == 0:\n            return proxy\n        return orig_imp(name, g, l, fromlist, level)\n    ctx = {'__name__': '__main__', '__file__': str(source), '__builtins__': {**vars(builtins), '__import__': imp}}\n    exec(compile(source.read_bytes(), str(source), 'exec'), ctx)\n", str(launcher), str(view)], cwd=tmp, env=env, check=True)
from zipfile import ZipFile
returned=Path('/kaggle/working/action_rerank/action_consistency_rerank.zip')
with ZipFile(returned) as z:
    assert z.testzip() is None and 'rerank_scores.json' in z.namelist() and 'paired_private.npz' in z.namelist()
print(returned, returned.stat().st_size, hashlib.sha256(returned.read_bytes()).hexdigest())
